# Caso C · 03 Features para detección de anomalías HVAC

> _Tutorial · Caso de uso: **C — Anomalías HVAC** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Construir features que separen normal de fallo: ΔT, duty cycles, ratios fan/valve, lags.


## 2. Qué se aprende

- ΔT como indicador de transferencia de calor.
- Duty cycle del HVAC (% on en una ventana).
- Ratio anomalía cuando válvula activa pero fan apagado.
- Por qué los autoencoders prefieren features escaladas.


## 3. Contexto del caso de uso

Las features físicamente significativas mejoran la interpretabilidad y reducen el espacio de búsqueda.


## 4. Relación con CENTINELA+

Mismas features se calcularían sobre `simarro-prod` cuando llegue un ticket de incidencia — para reproducir la firma.


## 5. Relación con Medallion

Lee plata, escribe oro local.


## 6. Datos de entrada

Plata mock (CSV) + etiquetas.


## 7. Schema CAPTIA esperado

No aplica para oro (parquet).


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos plata y etiquetas.


In [ ]:
csv_path = ROOT / "notebooks" / "_data" / "lbnl_fdd_rtu_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"]).sort_values("timestamp").set_index("timestamp")
df.head()


## 10. Exploración paso a paso

Computamos features y discutimos cobertura.


In [ ]:
def make_features(d):
    f = pd.DataFrame(index=d.index)
    f["dt_supply_return"] = d["RA_TEMP"] - d["SA_TEMP"]
    f["dt_supply_outdoor"] = d["OA_TEMP"] - d["SA_TEMP"]
    f["valve"] = d["CCV"]
    f["fan"] = d["FAN_STATE"]
    f["fan_valve_diff"] = f["valve"] - f["fan"]
    f["valve_duty_60"] = f["valve"].rolling("60min").mean()
    f["fan_duty_60"] = f["fan"].rolling("60min").mean()
    f["dt_lag_15"] = f["dt_supply_return"].shift(15)
    f["dt_change_15"] = f["dt_supply_return"] - f["dt_lag_15"]
    f["is_fault"] = d["is_fault"]
    f["fault_type"] = d["fault_type"]
    return f.dropna()

X = make_features(df)
X.head()


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Persistimos.


In [ ]:
out_dir = ROOT / "output" / "case_C"
out_dir.mkdir(parents=True, exist_ok=True)
parquet_path = out_dir / "hvac_features.parquet"
X.drop(columns=["fault_type"]).to_parquet(parquet_path)
print(f"Wrote {parquet_path.relative_to(ROOT)} ({len(X)})")


## 13. Visualizaciones explicativas

Distribución de `dt_supply_return` separada por `is_fault`.


In [ ]:
plot_distribution(X.assign(grupo=np.where(X.is_fault == 1, "fault", "normal")),
                  column="dt_supply_return", by="grupo", title="ΔT_return-supply normal vs fallo")


## 14. Validaciones

El target debe estar balanceado lo suficiente.


In [ ]:
counts = X["is_fault"].value_counts(normalize=True)
print(counts)
assert counts.min() > 0.02


## 15. Errores comunes

1. **Olvidar shift en rolling**: leakage.
2. **Usar lag mayor que ventana**: NaN al inicio.
3. **Mezclar fallos en mismo modelo binario sin codificar tipo**.


## 16. Ejercicios propuestos

1. Añade `valve_duty_15` y compara feature importance.
2. Discute si SHAP funcionaría mejor con o sin escalado.
3. Construye `fault_id` único por episodio.


## 17. Cómo se reutiliza con datos reales

`make_features` es pura: misma función sobre cualquier `simarro-prod` data.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `03_case_C_hvac_anomaly_detection/04_isolation_forest_autoencoder.ipynb`.
- Documento web del caso: `docs/use-cases/case-c-hvac-anomaly.md`.
